In [1]:
import rclpy
from rclpy.node import Node

from depthai_ros_msgs.msg import TrackDetection2DArray
from sensor_msgs.msg import Image

from cv_bridge import CvBridge

# Display images
from PIL import Image as PILImage
import numpy as np
from IPython.display import display, Image as IPImage, clear_output

import io
import matplotlib.pyplot as plt
import cv2

# TF
from geometry_msgs.msg import TransformStamped
from tf2_ros import TransformBroadcaster
from depthai_ros_msgs.msg import SpatialDetection

/usr/local/lib/python3.10/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
rclpy.init()

In [3]:
# rclpy.shutdown()

In [4]:
node = Node("check_oak_image") 

depthai_ros_msgs.msg.TrackDetection2DArray( 
header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1770190777, nanosec=963000234), frame_id='oak_rgb_camera_optical_frame'), detections=[ depthai_ros_msgs.msg.TrackDetection2D( results=[ vision_msgs.msg.ObjectHypothesisWithPose( hypothesis=vision_msgs.msg.ObjectHypothesis(class_id='0', score=0.30000001192092896), pose=geometry_msgs.msg.PoseWithCovariance( pose=geometry_msgs.msg.Pose( position=geometry_msgs.msg.Point(x=-0.09384978485107422, y=0.029244398117065428, z=0.5946548461914063), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)), covariance=array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.] ) ) )], bbox=vision_msgs.msg.BoundingBox2D( center=vision_msgs.msg.Pose2D(position=vision_msgs.msg.Point2D(x=114.0, y=178.5), theta=0.0), size_x=228.0, size_y=337.0), is_tracking=True, tracking_id='113', tracking_age=52, tracking_status=1)])

In [5]:
cv_image = None
bb_msg = None 


def show_image(image):
    # Convert to PIL Image
    pil_img = PILImage.fromarray(image)
    
    # Convert PIL image → PNG bytes
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    buf.seek(0)
    
    # Display
    display(IPImage(data=buf.getvalue()))

def draw_track_detections(image, msg):
    """
    image: OpenCV image (H x W x 3)
    msg: depthai_ros_msgs.msg.TrackDetection2DArray
    """
    det = msg

    obj_class = det.results[0].class_id

    if obj_class != 'person':
        return
        
    bbox = det.bbox

    # Center + size → corner coordinates
    cx = bbox.center.position.x
    cy = bbox.center.position.y
    w = bbox.size_x
    h = bbox.size_y

    x1 = int(cx - w / 2)
    y1 = int(cy - h / 2)
    x2 = int(cx + w / 2)
    y2 = int(cy + h / 2)

    # Draw rectangle
    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        color=(0, 255, 0),
        thickness=2
    )

    # Optional: label with tracking ID + score
    if det.results:
        hypothesis = det.results[0]
        label = f"ID {det.tracking_id} ({hypothesis.score:.2f})"

        cv2.putText(
            image,
            label,
            (x1, max(y1 - 5, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
            cv2.LINE_AA
        )

    return image


def stream_image(a, fmt='jpeg'):
    clear_output(wait=True)
    _,ret_array = cv2.imencode('.jpg', a) 
    i = IPImage(data=ret_array)
    display(i)
    
def detection_cb(msg):
    global bb_msg
    bb_msg = msg

    for det in bb_msg.detections:
        obj_class = det.results[0].hypothesis.class_id

        # if obj_class != '0':
        #     continue
            
        obj_track_id = det.tracking_id
        # print("### detection_cb ###")
        # print(msg)
        # print("### END detection_cb ###")
        t = TransformStamped()
    
        t.header.stamp = node.get_clock().now().to_msg()
        
        t.child_frame_id =  '{obj_class} [{track_id}]'.format(
            obj_class=obj_class,
            track_id=obj_track_id
        )
    
        tmp = det.results[0].pose.pose.position
        # t.header.frame_id = 'ir_omni'
        # t.transform.translation.x = tmp.z
        # t.transform.translation.y = tmp.x
        # t.transform.translation.z = tmp.y

        t.header.frame_id = 'oakd_rgb_camera_optical_frame'
        t.transform.translation.x = tmp.x
        t.transform.translation.y = tmp.y
        t.transform.translation.z = tmp.z
    
        tf_broadcaster.sendTransform(t)

def spatial_detection_cb(msg):
    global bb_msg
    bb_msg = msg

    det = msg
    obj_class = det.results[0].class_id

    if obj_class != 'person':
        bb_msg = None
        return
        
    obj_track_id = det.tracking_id
    # print("### detection_cb ###")
    # print(msg)
    # print("### END detection_cb ###")
    t = TransformStamped()

    t.header.stamp = node.get_clock().now().to_msg()
    
    t.child_frame_id =  '{obj_class}'.format(
        obj_class=obj_class
    )

    tmp = det.position
    # t.header.frame_id = 'ir_omni'
    # t.transform.translation.x = tmp.z
    # t.transform.translation.y = tmp.x
    # t.transform.translation.z = tmp.y

    t.header.frame_id = 'oakd_rgb_camera_optical_frame'
    t.transform.translation.x = tmp.x
    t.transform.translation.y = tmp.y
    t.transform.translation.z = tmp.z

    tf_broadcaster.sendTransform(t)


def image_cb(msg):
    global cv_image
    # print("### image_cb ###")
    bridge = CvBridge()
    cv_image = bridge.imgmsg_to_cv2(msg, desired_encoding='passthrough')
    
    # print("### END image_cb ###")
    

In [6]:
detection_sub = node.create_subscription(TrackDetection2DArray, '/track_detections', detection_cb, 10)

# detection_sub = node.create_subscription(SpatialDetection, '/color/yolov4_Spatial_detections', spatial_detection_cb, 10)

image_sub = node.create_subscription(Image, '/depthai/rgb/image', image_cb, 10)

tf_broadcaster = TransformBroadcaster(node)

In [7]:
rclpy.spin_once(node)

In [8]:
import time


# for _ in range(5000): 
while True:
    rclpy.spin_once(node)
    # if cv_image is not None:
    #     # im = cv2.cvtColor(cv_image, cv2.COLOR_BGR2RGB)

    #     if bb_msg is not None:
    #         im_with_bb = draw_track_detections(cv_image, bb_msg)
    #         stream_image(im_with_bb)
    #     else:
    #         stream_image(cv_image)
        # time.sleep(1/24)
    # plt.imshow(im)


KeyboardInterrupt



In [ ]:
bb_msg.detections[0].results[0].pose.pose.position

bb_msg.detections[0].tracking_id

In [ ]:
tmp = bb_msg.detections[0].results[0].pose.pose.position
tmp.x, tmp.y, tmp.z

In [ ]:
bb_msg.detections[0].results[0].hypothesis.class_id